Load schema definitions and config

In [0]:
%run ../config/config

In [0]:
dbutils.widgets.text("batch_id","")
batch_id=dbutils.widgets.get("batch_id")

In [0]:
bronze_table = f"{catalog}.{bronze_schema}.active_weather"
silver_table = f"{catalog}.{silver_schema}.active_weather"

Read bronze Delta table into a DataFrame

In [0]:
bronze_active_weather_df=(
    spark.read
    .format("delta")
    .table(bronze_table)
    )

Drop null records

In [0]:
bronze_active_weather_df= bronze_active_weather_df.dropna()

Drop duplicated records

In [0]:
bronze_active_weather_df=bronze_active_weather_df.dropDuplicates()

Select and rename columns for clarity and unification

In [0]:
from pyspark.sql import functions as F

silver_active_weather_df=(
    bronze_active_weather_df
    .select(
        F.col("status").alias("active_weather_key"),
        "weather_description",
        "batch_id"
    )
)

Add created and updated timestamp columns

In [0]:
silver_active_weather_df=(
    silver_active_weather_df
    .withColumns({
        "created_timestamp":F.current_timestamp(),
        "updated_timestamp":F.current_timestamp()
    })
)

Write DataFrame to silver Delta table

In [0]:
from pyspark.sql import Window

window = Window.partitionBy("active_weather_key").orderBy(
    F.col("batch_id").desc()
)

silver_active_weather_df = (
    silver_active_weather_df
    .withColumn("_rn", F.row_number().over(window))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

if not spark.catalog.tableExists(silver_table):
    silver_active_weather_df=(
        silver_active_weather_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(silver_table)
    )

else:
    from delta.tables import DeltaTable

    delta_table=DeltaTable.forName(spark, silver_table)
    (
        delta_table.alias("t")
        .merge(
            silver_active_weather_df.alias("s"),
            "t.active_weather_key = s.active_weather_key"
        )
        .whenMatchedUpdate(
            condition="s.batch_id >= t.batch_id",
            set={
                "active_weather_key": "s.active_weather_key",
                "weather_description": "s.weather_description",
                "batch_id": "s.batch_id",
                "updated_timestamp": "s.updated_timestamp"
            }

        )
        .whenNotMatchedInsertAll()
        .execute()
    )